In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from core.nn.timesfm import TimesFM_2p5_Model
from transformers import AutoTokenizer
from safetensors.torch import load_file
from core.nn.text_encoder import ModernBertModel
import torch

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    "deepvk/RuModernBERT-base", revision="patched-tokenizer"
)

In [3]:
timesfm = TimesFM_2p5_Model.from_pretrained(
    "/home/pomelk1n/Workspace/finam-forecast/models/timesFM_2p5"
)
timesfm

WARNING 10-04 17:23:12 [timesfm.py] Missing keys: ['stacked_xf.0.pre_crossattn_ln_ts.scale', 'stacked_xf.0.pre_crossattn_ln_text.scale', 'stacked_xf.0.post_crossattn_ln.scale', 'stacked_xf.0.cross_attn.kv_proj.weight', 'stacked_xf.0.cross_attn.q_proj.weight', 'stacked_xf.0.cross_attn.out.weight', 'stacked_xf.0.cross_attn.query_ln.scale', 'stacked_xf.0.cross_attn.key_ln.scale', 'stacked_xf.1.pre_crossattn_ln_ts.scale', 'stacked_xf.1.pre_crossattn_ln_text.scale', 'stacked_xf.1.post_crossattn_ln.scale', 'stacked_xf.1.cross_attn.kv_proj.weight', 'stacked_xf.1.cross_attn.q_proj.weight', 'stacked_xf.1.cross_attn.out.weight', 'stacked_xf.1.cross_attn.query_ln.scale', 'stacked_xf.1.cross_attn.key_ln.scale', 'stacked_xf.2.pre_crossattn_ln_ts.scale', 'stacked_xf.2.pre_crossattn_ln_text.scale', 'stacked_xf.2.post_crossattn_ln.scale', 'stacked_xf.2.cross_attn.kv_proj.weight', 'stacked_xf.2.cross_attn.q_proj.weight', 'stacked_xf.2.cross_attn.out.weight', 'stacked_xf.2.cross_attn.query_ln.scale', 's

TimesFM_2p5_Model(
  (text_encoder): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-

In [4]:
inputs = torch.stack(
    [
        torch.cat([torch.zeros(32), torch.randn(480)], dim=0),
        torch.cat([torch.zeros(2), torch.randn(510)], dim=0),
    ],
    dim=0,
)

mask = torch.stack(
    [
        torch.cat([torch.ones(32), torch.zeros(480)], dim=0),
        torch.cat([torch.ones(2), torch.zeros(510)], dim=0),
    ],
    dim=0,
)

In [7]:
inputs_text = tokenizer(
    ["deepvk/RuModernBERT-base", "привет"], return_tensors="pt", padding=True
)
inputs_text

{'input_ids': tensor([[50281,  6233,  1227,    92,    81,    21, 31046, 30526,  4578, 38212,
            58,    19, 37825, 50282],
        [50281,  1014,   904, 50282, 50283, 50283, 50283, 50283, 50283, 50283,
         50283, 50283, 50283, 50283]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

In [11]:
output = timesfm.forecast(inputs_ts=inputs, mask_ts=mask, inputs_text=inputs_text['input_ids'], mask_text=inputs_text['attention_mask'])

In [ ]:
output.shape

torch.Size([2, 16, 128])

In [ ]:
model = ModernBertModel.from_pretrained("deepvk/RuModernBERT-base")
state_dict = load_file(
    "/home/pomelk1n/Workspace/finam-forecast/models/timesfm/bertmlm.safetensors"
)
new_state_dict = {}
key2del = "model."
for k, v in state_dict.items():
    if k.startswith(key2del):
        new_key = k[len(key2del) :]
        new_state_dict[new_key] = v
    else:
        new_state_dict[k] = v
model.load_state_dict(new_state_dict, strict=False)

Some weights of ModernBertModel were not initialized from the model checkpoint at deepvk/RuModernBERT-base and are newly initialized: ['embeddings.no_news_token']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


_IncompatibleKeys(missing_keys=['embeddings.no_news_token'], unexpected_keys=['decoder.bias', 'head.dense.weight', 'head.norm.weight'])

In [ ]:
timesfm_state_dict = load_file(
    "/home/pomelk1n/Workspace/finam-forecast/models/timesfm/model.safetensors"
)

In [21]:
timesfm = TimesFM_2p5_Model.from_config(model.config.to_dict())
timesfm.load_state_dict(timesfm_state_dict, strict=False)

_IncompatibleKeys(missing_keys=['text_encoder.embeddings.no_news_token', 'text_encoder.embeddings.tok_embeddings.weight', 'text_encoder.embeddings.norm.weight', 'text_encoder.layers.0.attn.Wqkv.weight', 'text_encoder.layers.0.attn.Wo.weight', 'text_encoder.layers.0.mlp_norm.weight', 'text_encoder.layers.0.mlp.Wi.weight', 'text_encoder.layers.0.mlp.Wo.weight', 'text_encoder.layers.1.attn_norm.weight', 'text_encoder.layers.1.attn.Wqkv.weight', 'text_encoder.layers.1.attn.Wo.weight', 'text_encoder.layers.1.mlp_norm.weight', 'text_encoder.layers.1.mlp.Wi.weight', 'text_encoder.layers.1.mlp.Wo.weight', 'text_encoder.layers.2.attn_norm.weight', 'text_encoder.layers.2.attn.Wqkv.weight', 'text_encoder.layers.2.attn.Wo.weight', 'text_encoder.layers.2.mlp_norm.weight', 'text_encoder.layers.2.mlp.Wi.weight', 'text_encoder.layers.2.mlp.Wo.weight', 'text_encoder.layers.3.attn_norm.weight', 'text_encoder.layers.3.attn.Wqkv.weight', 'text_encoder.layers.3.attn.Wo.weight', 'text_encoder.layers.3.mlp_n

In [22]:
timesfm.text_encoder.load_state_dict(new_state_dict, strict=False)

_IncompatibleKeys(missing_keys=['embeddings.no_news_token'], unexpected_keys=['decoder.bias', 'head.dense.weight', 'head.norm.weight'])

In [ ]:
for (times_name, times_param), (model_name, model_param) in zip(
    timesfm.text_encoder.named_parameters(), model.named_parameters()
):
    if not torch.allclose(times_param, model_param):
        print(f"Parameter {times_name} is different from {model_name}")

Parameter embeddings.no_news_token is different from embeddings.no_news_token


In [24]:
new_temp_timesfm = TimesFM_2p5_Model.from_config(model.config.to_dict())
new_temp_timesfm.load_state_dict(timesfm_state_dict, strict=False)

_IncompatibleKeys(missing_keys=['text_encoder.embeddings.no_news_token', 'text_encoder.embeddings.tok_embeddings.weight', 'text_encoder.embeddings.norm.weight', 'text_encoder.layers.0.attn.Wqkv.weight', 'text_encoder.layers.0.attn.Wo.weight', 'text_encoder.layers.0.mlp_norm.weight', 'text_encoder.layers.0.mlp.Wi.weight', 'text_encoder.layers.0.mlp.Wo.weight', 'text_encoder.layers.1.attn_norm.weight', 'text_encoder.layers.1.attn.Wqkv.weight', 'text_encoder.layers.1.attn.Wo.weight', 'text_encoder.layers.1.mlp_norm.weight', 'text_encoder.layers.1.mlp.Wi.weight', 'text_encoder.layers.1.mlp.Wo.weight', 'text_encoder.layers.2.attn_norm.weight', 'text_encoder.layers.2.attn.Wqkv.weight', 'text_encoder.layers.2.attn.Wo.weight', 'text_encoder.layers.2.mlp_norm.weight', 'text_encoder.layers.2.mlp.Wi.weight', 'text_encoder.layers.2.mlp.Wo.weight', 'text_encoder.layers.3.attn_norm.weight', 'text_encoder.layers.3.attn.Wqkv.weight', 'text_encoder.layers.3.attn.Wo.weight', 'text_encoder.layers.3.mlp_n

In [ ]:
for (temp_times_name, temp_times_param), (timesfm_name, timesfm_param) in zip(
    new_temp_timesfm.named_parameters(), timesfm.named_parameters()
):
    if not torch.allclose(temp_times_param, timesfm_param):
        if "text_encoder" in temp_times_name:
            continue
        print(f"Parameter {temp_times_name} is different from {timesfm_name}")

Parameter text_projection.weight is different from text_projection.weight
Parameter text_projection.bias is different from text_projection.bias


In [29]:
timesfm.save_pretrained("/home/pomelk1n/Workspace/finam-forecast/models/timesFM_2p5")

In [ ]:
timesfm = TimesFM_2p5_Model.from_pretrained(
    "/home/pomelk1n/Workspace/finam-forecast/models/timesFM_2p5"
)
timesfm

WARNING 10-04 14:45:39 [timesfm.py] Unexpected keys: ['text_projection.bias']


TimesFM_2p5_Model(
  (text_encoder): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-

In [ ]:
inputs = torch.cat([torch.zeros(1, 32), torch.randn(1, 480)], dim=1)
mask = torch.cat([torch.ones(1, 32), torch.zeros(1, 480)], dim=1)

In [5]:
timesfm.forecast(inputs, mask)

tensor([[[ 0.1991,  0.3790,  0.4367,  ..., -0.7668, -0.6698, -0.8708],
         [-0.5996, -0.5018, -0.6218,  ..., -0.4383, -0.4927, -0.4887],
         [ 0.1528,  0.1259,  0.1069,  ..., -0.0024, -0.0253,  0.0075],
         ...,
         [-0.0924, -0.1073, -0.1191,  ..., -0.1139, -0.1181, -0.1054],
         [-0.0928, -0.0762, -0.0721,  ..., -0.0609, -0.0752, -0.0543],
         [-0.0475, -0.0519, -0.0551,  ..., -0.0377, -0.0520, -0.0314]]],
       grad_fn=<AddBackward0>)

In [8]:
timesfm.state_dict().keys()

odict_keys(['text_encoder.embeddings.no_news_token', 'text_encoder.embeddings.tok_embeddings.weight', 'text_encoder.embeddings.norm.weight', 'text_encoder.layers.0.attn.Wqkv.weight', 'text_encoder.layers.0.attn.Wo.weight', 'text_encoder.layers.0.mlp_norm.weight', 'text_encoder.layers.0.mlp.Wi.weight', 'text_encoder.layers.0.mlp.Wo.weight', 'text_encoder.layers.1.attn_norm.weight', 'text_encoder.layers.1.attn.Wqkv.weight', 'text_encoder.layers.1.attn.Wo.weight', 'text_encoder.layers.1.mlp_norm.weight', 'text_encoder.layers.1.mlp.Wi.weight', 'text_encoder.layers.1.mlp.Wo.weight', 'text_encoder.layers.2.attn_norm.weight', 'text_encoder.layers.2.attn.Wqkv.weight', 'text_encoder.layers.2.attn.Wo.weight', 'text_encoder.layers.2.mlp_norm.weight', 'text_encoder.layers.2.mlp.Wi.weight', 'text_encoder.layers.2.mlp.Wo.weight', 'text_encoder.layers.3.attn_norm.weight', 'text_encoder.layers.3.attn.Wqkv.weight', 'text_encoder.layers.3.attn.Wo.weight', 'text_encoder.layers.3.mlp_norm.weight', 'text_e

In [ ]:
timesfm = TimesFM_2p5_Model.from_pretrained(
    "/home/pomelk1n/Workspace/finam-forecast/models/timesFM_2p5"
)
timesfm

WARNING 10-04 12:08:43 [timesfm.py] Unexpected keys: ['text_projection.bias']


TimesFM_2p5_Model(
  (text_encoder): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-

In [ ]:
timesfm

In [ ]:
# language: python
from PyPDF2 import PdfReader

reader = PdfReader(
    "/home/pomelk1n/Downloads/Orlov_V_V_Osnovy_filosofii_Ch_1_Obschaya_filosofia_Vyp_1_Perm_2001.pdf"
)
text = []
for p in reader.pages:
    text.append(p.extract_text() or "")
full_text = "\n".join(text)
len(full_text)

576988

In [ ]:
num_tokens = 0
ptr = 0
news = []
tokens = []
while num_tokens < 10000:
    if 0.4 < random.random():
        ptr += 1
        news.append(None)
        tokens.append(None)
    else:
        ptr_right = random.randint(1, 2000)
        publication = full_text[ptr : ptr + ptr_right]
        ptr += ptr_right
        tokenized = tokenizer(publication, add_special_tokens=False)["input_ids"]
        num_tokens += len(tokenized)
        news.append(publication)
        tokens.append(tokenized)
len(news), num_tokens

(53, 10088)

In [24]:
tokens

[None,
 [20,
  418,
  20,
  26403,
  11681,
  418,
  205,
  45704,
  5912,
  25977,
  4,
  32030,
  11681,
  15588,
  32030,
  560,
  205,
  477,
  2314,
  109,
  537,
  1128,
  4,
  32030,
  11681,
  15588,
  32030,
  1128,
  205,
  3350,
  15985,
  4,
  23,
  205,
  205,
  6830,
  5154,
  5413,
  19829,
  45511,
  4,
  477,
  651,
  6559,
  19889,
  7710,
  19059,
  4,
  205,
  6184,
  37929,
  4107,
  22810,
  4107,
  4,
  860,
  669,
  10903,
  6559,
  42370,
  205,
  5099,
  618,
  1357,
  4,
  33659,
  3526,
  4,
  276,
  14805,
  4646,
  148,
  231,
  205,
  418,
  20,
  418,
  20,
  26403,
  11681,
  418,
  205,
  45704,
  5912,
  25977,
  4,
  32030,
  11681,
  15588,
  32030,
  560,
  205,
  41219,
  4,
  2411,
  1289,
  205,
  477,
  2314,
  109,
  537,
  1128,
  4,
  32030,
  11681,
  15588,
  32030,
  1128,
  205,
  3350,
  15985,
  4,
  23,
  205,
  44545,
  616,
  4,
  1557,
  8750,
  18,
  4,
  462,
  1211,
  20460,
  4,
  147,
  123,
  739,
  3161,
  2740,
  205,
  743

In [26]:
news

[None,
 '.В. ОРЛОВ\nОСНОВЫ  ФИЛОСОФИИ\nОБЩАЯ  ФИЛОСОФИЯ\nВыпуск  1\n\nМИНИСТЕРСТВО  ОБРАЗОВАНИЯ  \nРОССИЙСКОЙ  ФЕДЕРАЦИИ\nПермский  государственный  университет\nВ.В. ОРЛОВ\nОСНОВЫ  ФИЛОСОФИИ\nЧасть  первая\nОБЩАЯ  ФИЛОСОФИЯ\nВыпуск  1\nИздание  третье,  дополненное  и переработанное\nУчебное  пособие\nПермь  2001\n\nББК  15.1\n0-66\nПечатается  по постановлению  редакционно-издательского  совета  \nПермского  университета\nРецензенты:\nд-р филос.  н., проф.  Т.С: Васильева  \nд-р филос.  н., проф.  Н.Г. Магомедов\nОРЛОВ  В.В.\n0-66 Основы  философии.  Часть  первая.  Общая  философия.  Вып. 1 : Учебное  \nпособие.  3-еизд,  перераб.  и доп. Пермский  университет.-  Пермь.  2001..  - 216 с.\nISBN  5-7944-0194-Х\nУчебное  пособие  «Основы  философии»  состоит  из двух частей:  \n«Общая  философия»  и «Социальная  философия».\n«Общая  философия»  представлена  двумя выпусками  (соответс \xad\nтвенно,  темы 1-5,6-8).\nВ ней раскрывается  содержание  наиболее  общей  философской  на\xad\nу

In [ ]:
news_tokenizer = NewsTokenizerWrapper(no_news_token_id=model.config.no_news_token_id)

WARNING 10-04 04:36:45 [core.nn.text_encoder.tokenizer] Tokenizer max length 1000000000000000019884624838656 is not equal to requested max_length 8192. Setting to 8192.
WARNING 10-04 04:36:45 [core.nn.text_encoder.tokenizer] Tokenizer truncation side right is not left. Setting to left.


In [33]:
tokens = news_tokenizer.tokenize_news(news, return_tensors="pt")["input_ids"]
tokens

tensor([[50281, 16084,     4,  ..., 10651,   357, 50282]])

In [34]:
torch.any(tokens == news_tokenizer.no_news_token_id)

tensor(True)

In [35]:
bert_embedding = model.embeddings

In [40]:
embeddings = bert_embedding(tokens)
embeddings

tensor([[[-0.0010,  0.0732,  0.1546,  ...,  0.0585,  0.1385, -0.1346],
         [-0.3305,  0.4793, -0.6391,  ...,  0.1577, -0.4595,  0.1179],
         [ 0.3361,  0.0493,  0.2830,  ...,  0.0787,  0.1095,  0.3004],
         ...,
         [ 0.1369,  0.5441,  0.1585,  ..., -0.2912, -0.4404,  0.5389],
         [ 0.2881,  0.4320,  0.4718,  ..., -0.2462, -0.1840, -0.1883],
         [ 0.0140, -0.1664,  0.2180,  ..., -0.0140, -0.2840,  0.2161]]],
       grad_fn=<WhereBackward0>)

In [42]:
torch.all(embeddings == 0, dim=-1).any()

tensor(True)

In [71]:
tokens

[None,
 None,
 [418,
  20,
  26403,
  11681,
  418,
  205,
  45704,
  5912,
  25977,
  4,
  32030,
  11681,
  15588,
  32030,
  560,
  205,
  477,
  2314,
  109,
  537,
  1128,
  4,
  32030,
  11681,
  15588,
  32030,
  1128,
  205,
  3350,
  15985,
  4,
  23,
  205,
  205,
  6830,
  5154,
  5413,
  19829,
  45511,
  4,
  477,
  651,
  6559,
  19889,
  7710,
  19059,
  4,
  205,
  6184,
  37929,
  4107,
  22810,
  4107,
  4,
  860,
  669,
  10903,
  6559,
  42370,
  205,
  5099,
  618,
  1357,
  4,
  33659,
  3526,
  4,
  276,
  14805,
  4646,
  148,
  231,
  205,
  418,
  20,
  418,
  20,
  26403,
  11681,
  418,
  205,
  45704,
  5912,
  25977,
  4,
  32030,
  11681,
  15588,
  32030,
  560,
  205,
  41219,
  4,
  2411,
  1289,
  205,
  477,
  2314,
  109,
  537,
  1128,
  4,
  32030,
  11681,
  15588,
  32030,
  1128,
  205,
  3350,
  15985,
  4,
  23,
  205,
  44545,
  616,
  4,
  1557,
  8750,
  18,
  4,
  462,
  1211,
  20460,
  4,
  147,
  123,
  739,
  3161,
  2740,
  205,
  74

In [78]:
formatted_tokens = news_tokenizer.apply_format_to_tokens(tokens)
print(news_tokenizer.tokenizer.decode(formatted_tokens, skip_special_tokens=False))

[CLS]ситуации,  обусловленным  заметным  ист[SEP][unused0][SEP][unused0][SEP]ением  
ресурсов  Земли  и ее растущим  загрязнением.  По-прежнему  существу ­
ет также  опасность  уничтожения  человечества  в результате  термоядер ­
ной войны,  угроза  которой  в будущем  может  усилиться  в связи  с кри­
тическим  обострением  экологического  и демографического  кризиса.
Выяснено,  что при существующем  способе  существования  че­
ловечества,  основанном  на частной  собственности  и рынке,  по запад ­
ным стандартам  жизни  уже в обозримом  будущем  сможет  жить на 
Земле  лишь один, так называемый  «золотой»,  миллиард  людей,  в то 
время как сейчас  численность  человечества  приближается  к шести  
миллиардам.  Сможет  ли человечество  справиться  с этой, кажущейся  
неразрешимой,  проблемой?  Войны,  вымирание  миллиардов  людей,  
массовый  геноцид  могут  стать  реальностью  обозримого  будущего,  ес­
ли человечество  не найдет  другого  способа  социального  существова ­
ния, не

In [79]:
len(formatted_tokens)

8192

In [28]:
type(tokenizer)

transformers.tokenization_utils_fast.PreTrainedTokenizerFast

In [ ]:
tokenizer.to

In [16]:
tokenizer(full_text)

Token indices sequence length is longer than the specified maximum sequence length for this model (207384 > 8192). Running this sequence through the model will result in indexing errors


{'input_ids': [50281, 418, 20, 418, 20, 26403, 11681, 418, 205, 45704, 5912, 25977, 4, 32030, 11681, 15588, 32030, 560, 205, 477, 2314, 109, 537, 1128, 4, 32030, 11681, 15588, 32030, 1128, 205, 3350, 15985, 4, 23, 205, 205, 6830, 5154, 5413, 19829, 45511, 4, 477, 651, 6559, 19889, 7710, 19059, 4, 205, 6184, 37929, 4107, 22810, 4107, 4, 860, 669, 10903, 6559, 42370, 205, 5099, 618, 1357, 4, 33659, 3526, 4, 276, 14805, 4646, 148, 231, 205, 418, 20, 418, 20, 26403, 11681, 418, 205, 45704, 5912, 25977, 4, 32030, 11681, 15588, 32030, 560, 205, 41219, 4, 2411, 1289, 205, 477, 2314, 109, 537, 1128, 4, 32030, 11681, 15588, 32030, 1128, 205, 3350, 15985, 4, 23, 205, 44545, 616, 4, 1557, 8750, 18, 4, 462, 1211, 20460, 4, 147, 123, 739, 3161, 2740, 205, 743, 390, 16665, 4, 11939, 817, 205, 5099, 618, 285, 4, 27332, 205, 205, 651, 25968, 4, 3119, 20, 23, 205, 22, 19, 10506, 205, 5099, 426, 318, 564, 4, 430, 10051, 5920, 4, 36147, 272, 9307, 19, 803, 29010, 912, 4, 28733, 4, 205, 5099, 618, 912, 4,

In [27]:
tokenizer(full_text, truncation=True, return_tensors="pt")

{'input_ids': tensor([[50281,  7363, 10200,  ...,  6067,   205, 50282]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]])}

In [8]:
tokenizer.save_pretrained("/home/pomelk1n/Workspace/finam-forecast/models")

('/home/pomelk1n/Workspace/finam-forecast/models/tokenizer_config.json',
 '/home/pomelk1n/Workspace/finam-forecast/models/special_tokens_map.json',
 '/home/pomelk1n/Workspace/finam-forecast/models/tokenizer.json')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "/home/pomelk1n/Workspace/finam-forecast/models"
)

ImportError: 
 requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


In [10]:
tokenizer.truncation_side

'right'

In [ ]:
(-64) % 32

0

In [8]:
33 % 32

1

In [43]:
model = TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

In [44]:
model.compile(
    ForecastConfig(
        max_context=512,
        max_horizon=128,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

In [46]:
point_forecast, quantile_forecast = model.forecast(
    horizon=12,
    inputs=[
        np.linspace(0, 1, 29),
        np.sin(np.linspace(0, 20, 67)),
    ],  # Two dummy inputs
)